In [4]:
from functools import reduce
import time
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=150)

def cp_als(X, rank, max_iter=100, tol=1e-6, verbose=True):
    shape, N = X.shape, X.ndim

    #Initialize các ma trận A(n) (factor matrix) (n= 1,2,...,N)
    factors = [np.random.rand(d, rank) for d in shape]
    weights = np.ones(rank) # vector trọng số lambda có kích thước 1xR

    #Khatri-Rao product
    def khatri_rao(mats):
        return reduce(
            lambda A, B: np.einsum('ir,jr->jir', A, B).reshape(
                -1, rank, order='F'
            ), 
            mats,
        )
    norm_X = np.linalg.norm(X)
    prev_err = float('inf')
    errors = []
    start_time = time.time()

    if verbose:
        print(
            f"{'Lần lặp':<10} | {'Sai số tương đối':<35} | {'Mức suy giảm sai số':<25}"
        )
        print("-" * 75)

    # ALS loop
    for it in range(max_iter):
        for n in range(N):
            # Lấy các factor matrix ngoại trừ mode n theo thứ tự GIẢM DẦN: A^(N) ... A^(n+1) A^(n-1) ... A^(1)
            subs_desc = [
                factors[i] for i in reversed(range(N)) if i != n
            ]

            # V <- A^(1)T A^(1) * ... * A^(n-1)T A^(n-1) * A^(n+1)T A^(n+1) * ... * A^(N)T A^(N)
            V = reduce(np.multiply, [f.T @ f for f in subs_desc])

            # Unfolding X_(n)
            axes = [n] + [i for i in range(N) if i != n]
            X_n = np.transpose(X, axes).reshape(shape[n], -1, order='F')

            # A_hat <- X_(n) (A^(N) ... A^(n+1) A^(n-1) ... A^(1)) V^pseudoinverse
            KR_subs = khatri_rao(subs_desc)
            A_hat = X_n @ KR_subs @ np.linalg.pinv(V)

            # Chuẩn hóa các cột của A^(n) để lưu trong vector trọng số lambda
            weights = np.linalg.norm(A_hat, axis=0)
            factors[n] = A_hat / np.where(weights == 0, 1, weights)

        # Sai số tái tạo mô hình: X_hat = [[ lambda; A^(1), A^(2), ..., A^(N) ]] (= tổng chạy từ r=1 đến r=R của lambda_r o ar^(1) o ... o ar^(N))
        KR_all = khatri_rao([factors[i] for i in reversed(range(1, N))])
        X1_rec = (factors[0] * weights) @ KR_all.T
        X_rec = X1_rec.reshape(shape, order='F')

        err = np.linalg.norm(X - X_rec) / norm_X
        errors.append(err)

        delta_err = (
            abs(prev_err - err) if prev_err != float('inf') else 0.0
        )

        if verbose:
            print(
                f"Lặp {it+1:3d}/{max_iter:<4d} | {err:<35.6f} | {delta_err:<25.6e}"
            )

        # Kiểm tra điều kiện hội tụ
        if delta_err < tol and it > 0:
            if verbose:
                print(f"Hội tụ tại vòng lặp thứ {it+1}")
            break
        prev_err = err

    elapsed_time = time.time() - start_time

    if verbose:
        total_iters = len(errors)
        initial_err = errors[0] if errors else 0
        final_err = errors[-1] if errors else 0
        avg_speed = (
            (initial_err - final_err) / total_iters if total_iters > 0 else 0
        )

        print("-" * 75)
        print(f"  Thời gian tính toán : {elapsed_time:.4f} giây")
        print(f"  Số vòng lặp    : {total_iters}")
        print(f"  Mức giảm sai số TB / lặp : {avg_speed:.6e}")

    return weights, factors

In [8]:
shape_str = input("Kích thước tensor: ")
shape = tuple(map(int, shape_str.split(',')))
rank_str = input("Rank R (số lượng cột của factor matrix khi được optimized): ")
rank = int(rank_str)
np.random.seed(42)
X = np.random.rand(*shape)

print("\n" + "=" * 70)
print("=" * 70 + "\n")

print(" Frontal Slices (X[:, :, k]) ")
if X.ndim >= 3:
    K = shape[2]
    for k in range(K):
        slice_k = X[:, :, k] if X.ndim == 3 else X[:, :, k, ...]
        print(
            f"\nFrontal Slice k = {k+1} (Kích thước {slice_k.shape[0]} hàng x {slice_k.shape[1]} cột):"
        )
        print(slice_k)
else:
    print("\n(Tensor có số chiều < 3 nên toàn bộ Tensor là 1 ma trận):")
    print(X)

print("\n" + "=" * 70)
print(" Unfolded matrix X_(n) = [X_1  X_2  ...]:")
for n in range(X.ndim):
    axes = [n] + [i for i in range(X.ndim) if i != n]
    X_n = np.transpose(X, axes).reshape(shape[n], -1, order='F')
    print(
        f"\n (X_({n+1})) - {X_n.shape[0]} hàng x {X_n.shape[1]} cột:"
    )
    print(X_n)

print("\n" + "=" * 70)
print("=" * 70)
weights, factors = cp_als(X, rank=rank)
print("=" * 70 + "\n")

print(" Ma trận nhân tố kết quả:")
print(f"Vector trọng số Lambda (kích thước {len(weights)}):")
print(weights)
print("\n" + "-" * 50)

for i, factor in enumerate(factors):
    print(
        f"\nA^({i+1}) ({factor.shape[0]} hàng x {factor.shape[1]} cột ẩn):"
    )
    headers = "      " + "  ".join(
        [f"Comp {r+1}" for r in range(factor.shape[1])]
    )
    print(headers)

    for row_idx, row in enumerate(factor):
        row_str = "  ".join([f"{val:10.4f}" for val in row])
        print(f"Hàng {row_idx+1:2d}: {row_str}")
    print("-" * 50)

Kích thước tensor:  10,10,10
Rank R (số lượng cột của factor matrix khi được optimized):  5




 Frontal Slices (X[:, :, k]) 

Frontal Slice k = 1 (Kích thước 10 hàng x 10 cột):
[[0.3745 0.0206 0.6119 0.6075 0.122  0.9696 0.3887 0.7722 0.8631 0.1196]
 [0.0314 0.2898 0.8074 0.4174 0.9624 0.9083 0.3678 0.6776 0.3411 0.0931]
 [0.642  0.5487 0.6576 0.7948 0.9405 0.2944 0.615  0.8094 0.89   0.0305]
 [0.0517 0.439  0.5492 0.356  0.4916 0.5031 0.3882 0.1008 0.1182 0.9905]
 [0.1031 0.3193 0.7916 0.143  0.0848 0.7771 0.1175 0.0122 0.6294 0.4557]
 [0.6982 0.6134 0.5941 0.8129 0.9541 0.5202 0.7041 0.514  0.4591 0.1388]
 [0.1689 0.0393 0.1845 0.7133 0.0201 0.0869 0.3561 0.1734 0.8171 0.3237]
 [0.5326 0.7602 0.9383 0.1643 0.4627 0.1169 0.1517 0.1106 0.6939 0.1534]
 [0.7072 0.9848 0.8029 0.2246 0.0131 0.0671 0.7755 0.3733 0.2311 0.7223]
 [0.2079 0.5903 0.8685 0.1394 0.2741 0.6005 0.0828 0.6637 0.3342 0.7994]]

Frontal Slice k = 2 (Kích thước 10 hàng x 10 cột):
[[0.9507 0.9699 0.1395 0.1705 0.4952 0.7751 0.2713 0.1987 0.6233 0.7132]
 [0.6364 0.1612 0.8961 0.2221 0.2518 0.2396 0.6323 0.0166 0.